# AI Engine Manager (AIEM) for managing AI engines

This notebook is intended to help you get started managing your AI Engines in the H2O AI Cloud using python. 

* **Product Documentation:** https://docs.h2o.ai/ai-engine-manager/
* **Python Documentation:** https://docs.h2o.ai/ai-engine-manager/py/installation

## Prerequistes

Update the `h2o_ai_cloud.py` file with the connection parameters for your H2O AI Cloud environemnt:
1. Login to your H2O AI Cloud environment
1. Click your username or avatar in the H2O AI Cloud navigation bar
1. Navigate to `CLI & API Access`
1. Use the variables from the `Accessing H2O AI Cloud APIs` section to populate the parameters

In [2]:
from h2o_ai_cloud import ai_engine_client

## Securely connect

We first connect to the H2O AI Cloud using our platform token to create a token provider object. We can then use this object to log into AI Engine Manager and other APIs.

In [ ]:
aiem_client = ai_engine_client()

Visit https://internal.dedicated.h2o.ai/auth/get-platform-token to get your platform token


In [5]:
import h2o_engine_manager
print(h2o_engine_manager.__version__)

1.10.0


## Managing Driverless AI

### View available versions

In [18]:
[dai_version.name for dai_version in aiem_client.dai_engine_version_client.list_all_dai_engine_versions(parent="workspaces/global")]

['workspaces/global/daiEngineVersions/2.5.0',
 'workspaces/global/daiEngineVersions/2.4.5',
 'workspaces/global/daiEngineVersions/2.4.4',
 'workspaces/global/daiEngineVersions/2.4.3',
 'workspaces/global/daiEngineVersions/2.4.2',
 'workspaces/global/daiEngineVersions/2.4.1',
 'workspaces/global/daiEngineVersions/2.3.2',
 'workspaces/global/daiEngineVersions/2.2.4',
 'workspaces/global/daiEngineVersions/1.11.1.1']

### View available profiles

* List the profiles
* View the options for max run time in this profile

In [ ]:
aiem_client.dai_engine_profile_client.list_all_assigned_dai_engine_profiles(parent="workspaces/global")

In [38]:
aiem_client.dai_engine_profile_client.list_all_assigned_dai_engine_profiles(parent="workspaces/global")[0].max_running_duration_constraint

{'default': '8h', 'maximum': '12h', 'minimum': '1h'}

### Create an engine in Personal Workspace (default)
See the `3 AutoML` tutorial for what to do once you've connected to the AutoML platform. 

In [100]:
dai_engine_client = aiem_client.dai_engine_client

In [ ]:
dai_engine = dai_engine_client.create_engine(
    cpu=8,
    gpu=0,
    memory_bytes="16Gi",
    storage_bytes="250G",
    max_idle_duration="3600s",
    max_running_duration ="36000s",
    display_name="My Engine",
    dai_engine_version='workspaces/global/daiEngineVersions/2.5.0'
)

In [51]:
dai_engine.wait()
dai = dai_engine.connect()

In [54]:
dai_engine.login_url

'https://enginemanager.internal.dedicated.h2o.ai/workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/my-engine-9039/oidc/login'

### List my instances

In [103]:
[engine.name for engine in dai_engine_client.list_all_engines()]

['workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/new-dai-engine-9161',
 'workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/my-engine-9039',
 'workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/my-engine',
 'workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/new-dai-engine-5830',
 'workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/new-dai-engine',
 'workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/new-dai-engine-7853']

### Download engine logs

In [ ]:
my_dai_engine = dai_engine_client.get_engine(engine_id="new-dai-engine1")
my_dai_engine.resume()
my_dai_engine.wait()
my_dai_engine.connect()

In [ ]:
my_dai_engine

In [115]:
with open("dai_engine_logs.txt", "a") as f:
  f.write(my_dai_engine.download_logs())

### Pause an engine

You can pause an engine that is currently running. Pausing an engine shuts it down to save resources, it is similar to powering off a server. You will not loose any data and you can start an engine at any time.

In [121]:
my_dai_engine.pause()

### Resume the engine

You can resume a paused engine by running:

In [122]:
my_dai_engine.resume()

my_dai_engine.state.value

'STATE_STARTING'

In [123]:
my_dai_engine.wait()

my_dai_engine.state.value

'STATE_RUNNING'

### Delete the engine

When you no longer need an engine, you can terminate it. Once deleted, there is no way to restart the engine or access any data.

In [126]:
my_dai_engine.delete()

### Check all DAI Engines state

In [131]:
# it takes ~20 seconds for the status to update
for dai_engine in dai_engine_client.list_all_engines():
    print(dai_engine.name, "\t", dai_engine.state.value, "\t",
          dai_engine.workspace_id, "\t", dai_engine.create_time, "\t", dai_engine.uid)

workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/new-dai-engine-9161 	 STATE_RUNNING 	 f6b5a96b-b3bf-4ec1-b495-868d52045e05 	 2026-09-23 12:02:20+00:00 	 be7064c9-f87b-4cde-81fd-5f805d38aa28
workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/my-engine-9039 	 STATE_PAUSED 	 f6b5a96b-b3bf-4ec1-b495-868d52045e05 	 2026-09-23 10:33:40+00:00 	 1890c94f-fb6a-487d-a04c-5849911aedb9
workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/my-engine 	 STATE_RUNNING 	 f6b5a96b-b3bf-4ec1-b495-868d52045e05 	 2026-09-23 10:31:31+00:00 	 3b7d970e-65ed-4beb-8a8e-bba80c53cab3
workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/new-dai-engine-5830 	 STATE_PAUSED 	 f6b5a96b-b3bf-4ec1-b495-868d52045e05 	 2026-09-23 08:04:23+00:00 	 b4db72b0-166b-4ecd-b3d9-9bf939dffe66
workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/daiEngines/new-dai-engine-7853 	 STATE_FAILED 	 f6b5a96b-b3bf-4ec1-b495-868d52045e05 	 2026-09-22 16:10:06+00:00 	 5d05cb01-a516-415e-b58c-4acdfa3e48b1


## Managing H2O-3

### View available versions

In [133]:
[h2o_version.name for h2o_version in aiem_client.h2o_engine_version_client.list_all_h2o_engine_versions(parent="workspaces/global")]

['workspaces/global/h2oEngineVersions/3.46.0.11-3',
 'workspaces/global/h2oEngineVersions/3.46.0.10-3',
 'workspaces/global/h2oEngineVersions/3.46.0.10-1',
 'workspaces/global/h2oEngineVersions/3.46.0.9-1',
 'workspaces/global/h2oEngineVersions/3.46.0.7']

### View available profiles

In [ ]:
aiem_client.h2o_engine_profile_client.list_all_assigned_h2o_engine_profiles(parent="workspaces/global")

In [135]:
aiem_client.h2o_engine_profile_client.list_all_assigned_h2o_engine_profiles(parent="workspaces/global")[0].max_running_duration_constraint

{'default': '8h', 'maximum': '12h', 'minimum': '1h'}

### Create a cluster
See the `4 H2O-3` Tutorial for what to do once you've connected to the distributed ML platform. 

In [138]:
h2o_engine_client = aiem_client.h2o_engine_client

In [140]:
h2o_engine = h2o_engine_client.create_engine(
    node_count= 1,
    cpu=8,
    gpu=0,
    memory_bytes="16Gi",
    max_idle_duration="3600s",
    max_running_duration ="36000s",
    display_name="My Engine",
    h2o_engine_version='workspaces/global/h2oEngineVersions/3.46.0.11-3'
)

In [143]:
h2o_engine.wait()

In [146]:
print("https://" + h2o_engine.get_connection_config()["ip"] + "/" + h2o_engine.get_connection_config()["context_path"] + "/flow/index.html")

https://enginemanager.internal.dedicated.h2o.ai/workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/h2oEngines/my-engine/flow/index.html


### List my clusters

In [147]:
[engine.name for engine in h2o_engine_client.list_all_engines()]

['workspaces/f6b5a96b-b3bf-4ec1-b495-868d52045e05/h2oEngines/my-engine']

### Download cluster logs

In [148]:
with open("h2o_engine_logs.txt", "a") as f:
  f.write(h2o_engine.download_logs())

### Stop an engine
H2O clusters cannot be resumed after they are stopped. Only turn off your cluster once you are permanately done with it.

In [149]:
h2o_engine.terminate()

In [150]:
print(f"State: {h2o_engine.state.value}")

State: STATE_TERMINATED


### Delete an engine

In [152]:
h2o_engine.delete()

## Managing Notebooks

In [173]:
notebook_engine_client = aiem_client.notebook_engine_client
notebook_engine_profile_client = aiem_client.notebook_engine_profile_client
notebook_engine_image_client = aiem_client.notebook_engine_image_client

# You can list available profiles.
profiles = notebook_engine_profile_client.list_all_assigned_notebook_engine_profiles(parent="workspaces/global")
print("Profiles:")
for profile in profiles:
    print("\t", profile.name)

# You can list available images.
images = notebook_engine_image_client.list_all_notebook_engine_images(parent="workspaces/global")
print("Images:")
for image in images:
    print("\t", image.name)

Profiles:
	 workspaces/global/notebookEngineProfiles/default
Images:
	 workspaces/global/notebookEngineImages/cpu-kernel


In [186]:
from h2o_engine_manager.clients.notebook_engine.engine import NotebookEngine

notebook_engine = notebook_engine_client.create_notebook_engine(
    parent="workspaces/global",  # You can use alias 'default' for your personal workspace.
    notebook_engine=NotebookEngine(
        profile=profiles[0].name,  # One of the listed profiles.
        notebook_image=images[0].name,  # One of the listed images.
        display_name="My Basic Notebook engine",
        cpu=4,
        gpu=0,
        memory_bytes="16Gi",
        storage_bytes="64Gi",
        max_idle_duration="1h",
        max_running_duration="12h",
    ),
    notebook_engine_id="my-basic-engine12345"
)

In [187]:
notebook_engine_client.wait(name=notebook_engine.name)

{'cpu': 4,
 'create_time': datetime.datetime(2026, 9, 23, 13, 29, 6, 815303, tzinfo=tzutc()),
 'creator': 'users/c19820dc-37ef-4d4f-893d-5f46492e51a0',
 'creator_display_name': 'audrey.leteve@h2o.ai',
 'current_idle_duration': None,
 'current_running_duration': None,
 'delete_time': None,
 'display_name': 'My Basic Notebook engine',
 'failure_reason': <NotebookEngineFailureReason.FAILURE_REASON_UNSPECIFIED: 'FAILURE_REASON_UNSPECIFIED'>,
 'gpu': 0,
 'max_idle_duration': '1h',
 'max_running_duration': '12h',
 'memory_bytes': '16Gi',
 'name': 'workspaces/global/notebookEngines/my-basic-engine12345',
 'notebook_image': 'workspaces/global/notebookEngineImages/cpu-kernel',
 'notebook_image_info': <h2o_engine_manager.clients.notebook_engine.notebook_engine_image_info.NotebookEngineImageInfo object at 0x11937fc90>,
 'profile': 'workspaces/global/notebookEngineProfiles/default',
 'profile_info': <h2o_engine_manager.clients.notebook_engine.notebook_engine_profile_info.NotebookEngineProfileInfo 

### List existing Notebook Engine from the default workspace

In [ ]:
all_engines = notebook_engine_client.list_all_notebook_engines(parent="workspaces/global")
all_engines

### Get Existing Notebook Engine

In [191]:
engine = notebook_engine_client.get_notebook_engine(name="workspaces/global/notebookEngines/my-basic-engine1")

print(engine)

{'cpu': 4,
 'create_time': datetime.datetime(2026, 9, 23, 13, 23, 51, 38911, tzinfo=tzutc()),
 'creator': 'users/c19820dc-37ef-4d4f-893d-5f46492e51a0',
 'creator_display_name': 'audrey.leteve@h2o.ai',
 'current_idle_duration': None,
 'current_running_duration': None,
 'delete_time': None,
 'display_name': 'My Basic Notebook engine',
 'failure_reason': <NotebookEngineFailureReason.FAILURE_REASON_UNSPECIFIED: 'FAILURE_REASON_UNSPECIFIED'>,
 'gpu': 0,
 'max_idle_duration': '1h',
 'max_running_duration': '8h',
 'memory_bytes': '14Gi',
 'name': 'workspaces/global/notebookEngines/my-basic-engine1',
 'notebook_image': 'workspaces/global/notebookEngineImages/cpu-kernel',
 'notebook_image_info': <h2o_engine_manager.clients.notebook_engine.notebook_engine_image_info.NotebookEngineImageInfo object at 0x1194ee7d0>,
 'profile': 'workspaces/global/notebookEngineProfiles/default',
 'profile_info': <h2o_engine_manager.clients.notebook_engine.notebook_engine_profile_info.NotebookEngineProfileInfo objec

### Pause Notebook Engine

In [ ]:
notebook_engine_client.pause_notebook_engine(name="workspaces/global/notebookEngines/my-basic-engine1234")

# You may wait for the engine to get into PAUSED state.
# This is a blocking call.
notebook_engine_client.wait(name="workspaces/global/notebookEngines/my-basic-engine1234")

### Update Notebook Engine Configs 

Update: CPU, Memory, idle duration and share with other users who have access to the workspace 

**Please note some changes require the notebook to be paused.**

In [ ]:
# Get the engine first.
engine = notebook_engine_client.get_notebook_engine(name="workspaces/global/notebookEngines/my-basic-engine1234")

# Change selected engine fields to new values.
engine.cpu = 4
engine.memory_bytes = "16Gi"
engine.max_idle_duration = "3600s"
engine.shared = True

# Perform the update.
notebook_engine_client.update_notebook_engine(notebook_engine=engine)

### Resume a paused engine.

In [ ]:
notebook_engine_client.resume_notebook_engine(name="workspaces/global/notebookEngines/my-basic-engine1234")

# You may wait for the engine to get into RUNNING state.
# It may take a few minutes.
notebook_engine_client.wait(name="workspaces/global/notebookEngines/my-basic-engine1234")

# You can access the again-running Notebook engine in browser via the access URI.
access_uri = notebook_engine_client.access_notebook_engine(name="workspaces/global/notebookEngines/my-basic-engine1234")
print("Notebook engine access URI: ", access_uri)

### Delete engine.

In [ ]:
# This returns immediately and does not wait for the engine to be deleted.
notebook_engine_client.delete_notebook_engine(name="workspaces/global/notebookEngines/my-basic-engine1234")

# You may wait for the engine to finish deleting.
notebook_engine_client.wait(name="workspaces/global/notebookEngines/my-basic-engine1234")

## Clean up

In [153]:
!rm dai_engine_logs.txt
!rm h2o_engine_logs.txt